In [ ]:
!pip install -U transformers accelerate datasets scikit-learn pandas matplotlib


In [ ]:
import os
import pickle
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_recall_fscore_support

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/ML_Models/emotion_model_v2"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
emotion_cols = [
    'admiration','amusement','anger','annoyance','approval','caring',
    'confusion','curiosity','desire','disappointment','disapproval',
    'disgust','embarrassment','excitement','fear','gratitude','grief',
    'joy','love','nervousness','optimism','pride','realization',
    'relief','remorse','sadness','surprise','neutral'
]


In [ ]:
dataset = load_dataset("go_emotions")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
df = dataset["train"].to_pandas().sample(
    n=12000,
    random_state=42
).reset_index(drop=True)


In [ ]:
label_names = dataset["train"].features["labels"].feature.names
print(label_names)


['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [ ]:
df["label_names"] = df["labels"].apply(
    lambda ids: [label_names[i] for i in ids]
)

df = df[df["label_names"].map(len) > 0].reset_index(drop=True)


In [ ]:
mlb = MultiLabelBinarizer(classes=label_names)
Y = mlb.fit_transform(df["label_names"])

with open(f"{MODEL_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(mlb, f)


In [ ]:
label_counts = Y.sum(axis=0)
label_counts = torch.tensor(label_counts, dtype=torch.float)

print("How often each emotion appears:")
for name, count in zip(label_names, label_counts):
    print(f"{name:15s}: {int(count)}")


How often each emotion appears:
admiration     : 1148
amusement      : 614
anger          : 432
annoyance      : 669
approval       : 817
caring         : 296
confusion      : 362
curiosity      : 627
desire         : 179
disappointment : 348
disapproval    : 549
disgust        : 211
embarrassment  : 77
excitement     : 237
fear           : 165
gratitude      : 759
grief          : 20
joy            : 425
love           : 592
nervousness    : 48
optimism       : 444
pride          : 35
realization    : 320
relief         : 39
remorse        : 150
sadness        : 377
surprise       : 288
neutral        : 3898


In [ ]:
class_weights = 1.0 / (label_counts + 1e-6)
class_weights = class_weights / class_weights.mean()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = class_weights.to(device)

print("\nImportance given to each emotion:")
for name, weight in zip(label_names, class_weights):
    print(f"{name:15s}: {weight.item():.2f}")



Importance given to each emotion:
admiration     : 0.12
amusement      : 0.23
anger          : 0.32
annoyance      : 0.21
approval       : 0.17
caring         : 0.47
confusion      : 0.38
curiosity      : 0.22
desire         : 0.77
disappointment : 0.40
disapproval    : 0.25
disgust        : 0.66
embarrassment  : 1.80
excitement     : 0.58
fear           : 0.84
gratitude      : 0.18
grief          : 6.92
joy            : 0.33
love           : 0.23
nervousness    : 2.88
optimism       : 0.31
pride          : 3.95
realization    : 0.43
relief         : 3.55
remorse        : 0.92
sadness        : 0.37
surprise       : 0.48
neutral        : 0.04


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

encodings = tokenizer(
    df["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }


In [ ]:
full_dataset = EmotionDataset(encodings, Y)

indices = np.arange(len(full_dataset))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.1,
    random_state=42
)

train_ds = torch.utils.data.Subset(full_dataset, train_idx)
val_ds   = torch.utils.data.Subset(full_dataset, val_idx)

print("Train:", len(train_ds))
print("Val:", len(val_ds))


Train: 10800
Val: 1200


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
param_dtype = next(model.parameters()).dtype
class_weights = class_weights.to(dtype=param_dtype, device=device)

class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_function = torch.nn.BCEWithLogitsLoss(
            pos_weight=class_weights
        )

        loss = loss_function(logits, labels)

        return (loss, outputs) if return_outputs else loss


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.25).astype(int)

    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="micro", zero_division=0
    )

    macro_f1 = f1_score(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "precision_micro": micro_p,
        "recall_micro": micro_r
    }


In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none",
)


In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Precision Micro,Recall Micro
1,0.062897,0.064731,0.000000,0.000000,0.000000,0.000000
2,0.052917,0.049404,0.159900,0.175713,0.790123,0.088951
3,0.041948,0.043874,0.268182,0.307207,0.735202,0.164003
4,0.034244,0.042807,0.328855,0.352256,0.725118,0.212648
5,0.029552,0.042203,0.361310,0.375296,0.685437,0.245309
6,0.027614,0.042745,0.370597,0.382188,0.698077,0.252259


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=8100, training_loss=0.046221411875736564, metrics={'train_runtime': 418.5831, 'train_samples_per_second': 154.808, 'train_steps_per_second': 19.351, 'total_flos': 2146966887628800.0, 'train_loss': 0.046221411875736564, 'epoch': 6.0})

In [ ]:
def predict(text, top_k=8):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits)[0].cpu().numpy()
    order = probs.argsort()[::-1]

    best_emotion = (
        label_names[order[0]],
        round(float(probs[order[0]]), 4)
    )

    emotion_probabilities = [
        (label_names[i], round(float(probs[i]), 4))
        for i in order[:top_k]
        if probs[i] > 0.01
    ]

    return {
        "best_emotion": best_emotion,
        "emotion_probabilities": emotion_probabilities
    }


In [ ]:
tests = [
    "I'm broken and sad",
    "I'm proud and excited",
    "I love you but I'm scared",
    "This is hilarious",
    "Everything goes wrong"
]

for t in tests:
    print(t)
    print(predict(t))
    print("-" * 40)


I'm broken and sad
{'best_emotion': ('sadness', 0.7448), 'emotion_probabilities': [('sadness', 0.7448), ('disappointment', 0.0554), ('remorse', 0.0255), ('grief', 0.0255), ('nervousness', 0.0136), ('fear', 0.013)]}
----------------------------------------
I'm proud and excited
{'best_emotion': ('pride', 0.906), 'emotion_probabilities': [('pride', 0.906), ('joy', 0.1261), ('admiration', 0.0911), ('relief', 0.0885), ('excitement', 0.0534), ('approval', 0.0373), ('surprise', 0.0318), ('annoyance', 0.0293)]}
----------------------------------------
I love you but I'm scared
{'best_emotion': ('fear', 0.6856), 'emotion_probabilities': [('fear', 0.6856), ('nervousness', 0.0501), ('love', 0.0165), ('desire', 0.0155), ('sadness', 0.0141), ('annoyance', 0.0108)]}
----------------------------------------
This is hilarious
{'best_emotion': ('amusement', 0.491), 'emotion_probabilities': [('amusement', 0.491), ('joy', 0.0262), ('excitement', 0.0135), ('surprise', 0.0114)]}
--------------------------

In [ ]:
# Save the best trained model
trainer.save_model(MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
tokenizer.save_pretrained(MODEL_DIR)


('/content/drive/MyDrive/ML_Models/emotion_model_v2/tokenizer_config.json',
 '/content/drive/MyDrive/ML_Models/emotion_model_v2/tokenizer.json')

In [ ]:
with open(f"{MODEL_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(mlb, f)


In [ ]:
import os

print("Saved files:")
for file in os.listdir(MODEL_DIR):
    print(file)


Saved files:
label_encoder.pkl
checkpoint-5400
checkpoint-6750
checkpoint-8100
config.json
model.safetensors
training_args.bin
tokenizer_config.json
tokenizer.json


In [5]:
import os

MODEL_DIR = "/content/drive/MyDrive/ML_Models/emotion_model_v2"

if os.path.exists(MODEL_DIR):
    print("✅ Model directory exists\n")
    print("📂 Files inside MODEL_DIR:\n")

    for file in os.listdir(MODEL_DIR):
        print(" -", file)
else:
    print("❌ Model directory does NOT exist")


❌ Model directory does NOT exist


In [12]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
import pickle

MODEL_DIR = "/content/drive/MyDrive/ML_Models/psysense-emotion-ai"

# Load model
model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)

# Load label encoder
with open(f"{MODEL_DIR}/label_encoder.pkl", "rb") as f:
    mlb = pickle.load(f)

print("✅ Model, tokenizer, and label encoder loaded successfully!")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model, tokenizer, and label encoder loaded successfully!


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultiLabelBinarizer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import torch
import pickle
import numpy as np
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast
)

# Path where model is saved
MODEL_DIR = "/content/drive/MyDrive/ML_Models/psysense-emotion-ai"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)

# Load label encoder
with open(f"{MODEL_DIR}/label_encoder.pkl", "rb") as f:
    mlb = pickle.load(f)

# Emotion labels
label_names = mlb.classes_

print("✅ Model, tokenizer, and labels loaded")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model, tokenizer, and labels loaded


In [14]:
def predict_emotions(text, threshold=0.25, top_k=8):
    """
    Predict emotions for a given text.
    Returns structured output (industry-friendly).
    """

    # Safety check
    if not text or not text.strip():
        return {"error": "Empty input text"}

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    # Forward pass
    with torch.no_grad():
        logits = model(**inputs).logits

    # Convert logits → probabilities
    probs = torch.sigmoid(logits)[0].cpu().numpy()

    # Sort by confidence
    sorted_idx = probs.argsort()[::-1]

    # Dominant emotion
    dominant_emotion = {
        "label": label_names[sorted_idx[0]],
        "confidence": round(float(probs[sorted_idx[0]]), 4)
    }

    # Active emotions (above threshold)
    active_emotions = [
        {
            "label": label_names[i],
            "confidence": round(float(probs[i]), 4)
        }
        for i in sorted_idx
        if probs[i] >= threshold
    ]

    # Top-k (transparent view)
    top_emotions = [
        (label_names[i], round(float(probs[i]), 4))
        for i in sorted_idx[:top_k]
    ]

    return {
        "input_text": text,
        "dominant_emotion": dominant_emotion,
        "active_emotions": active_emotions,
        "top_emotions": top_emotions
    }


In [15]:
tests = [
    "I'm broken and sad",
    "I'm proud and excited",
    "I love you but I'm scared",
    "This is hilarious",
    "Everything goes wrong",
    "ok",
    "🙂"
]

for t in tests:
    result = predict_emotions(t)
    print(result)
    print("-" * 50)


{'input_text': "I'm broken and sad", 'dominant_emotion': {'label': 'sadness', 'confidence': 0.7448}, 'active_emotions': [{'label': 'sadness', 'confidence': 0.7448}], 'top_emotions': [('sadness', 0.7448), ('disappointment', 0.0554), ('remorse', 0.0255), ('grief', 0.0255), ('nervousness', 0.0137), ('fear', 0.013), ('caring', 0.009), ('realization', 0.0082)]}
--------------------------------------------------
{'input_text': "I'm proud and excited", 'dominant_emotion': {'label': 'pride', 'confidence': 0.9061}, 'active_emotions': [{'label': 'pride', 'confidence': 0.9061}], 'top_emotions': [('pride', 0.9061), ('joy', 0.126), ('admiration', 0.091), ('relief', 0.0885), ('excitement', 0.0533), ('approval', 0.0373), ('surprise', 0.0318), ('annoyance', 0.0292)]}
--------------------------------------------------
{'input_text': "I love you but I'm scared", 'dominant_emotion': {'label': 'fear', 'confidence': 0.6856}, 'active_emotions': [{'label': 'fear', 'confidence': 0.6856}], 'top_emotions': [('f